<a href="https://colab.research.google.com/github/chadallen/insider_trading_detection/blob/main/Detect_insider_trading_on_prediction_markets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook looks for suspicious trading patterns in resolved Polymarket prediction markets.

**How it works**
1. We pull ~120 recently resolved markets and score each one across four signals:

2. VPIN — Models for detecting insider trading in
*financial markets* (Easley, López de Prado & O'Hara, 2011, 2012)
3. Time-weighted VPIN — same, but weighted toward moves closer to resolution
4. Resolution surprise — how wrong was the market's implied probability vs. what actually happened?
5. Late move ratio — what % of total price movement happened right before resolution?
6. We then run Isolation Forest across all four signals to rank markets by how anomalous they look.

Note: Polymarket only gives us 12h price granularity for closed markets, so VPIN is approximated from price movement rather than actual order flow.
No labeled training data yet — this is unsupervised
~120 markets is a small sample



In [56]:
#@title 1 - Install dependencies

!pip install requests pandas numpy scikit-learn plotly -q

In [57]:
#@title 2 - Fetch resolved markets

import requests
import pandas as pd
import json
from datetime import datetime, timedelta, timezone

def fetch_resolved_markets(days_back=365, limit=200):
    """Fetch recently resolved markets with token IDs included."""

    end_date = datetime.now(timezone.utc)
    start_date = end_date - timedelta(days=days_back)

    url = "https://gamma-api.polymarket.com/markets"
    params = {
        "closed": "true",
        "limit": limit,
        "after": start_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "order": "endDate",
        "ascending": "false",
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    markets = []
    for m in data:
        # parse token IDs
        raw_tokens = m.get("clobTokenIds", "[]")
        try:
            tokens = json.loads(raw_tokens) if isinstance(raw_tokens, str) else raw_tokens
            token_id = tokens[0] if tokens else None
        except:
            token_id = None

        markets.append({
            "market_id":       m.get("conditionId"),
            "token_id":        token_id,
            "question":        m.get("question"),
            "resolution_time": m.get("endDate"),
            "final_price":     m.get("outcomePrices"),
            "volume":          float(m.get("volume") or 0),
        })

    df = pd.DataFrame(markets)
    # drop rows with no token_id or no volume
    df = df[df["token_id"].notna() & (df["volume"] > 0)]
    print(f"Fetched {len(df)} markets with trade data")
    return df

df_markets = fetch_resolved_markets(days_back=548)
print(df_markets[["question", "volume", "token_id"]].head(5))

Fetched 182 markets with trade data
                                        question    volume  \
0     idOS FDV above $100M one day after launch?  9606.287   
1      idOS FDV above $50M one day after launch? 21937.653   
2  Espresso FDV above $50M one day after launch? 83280.458   
3   Espresso FDV above $1B one day after launch? 79468.370   
4      idOS FDV above $80M one day after launch? 12133.768   

                                                      token_id  
0  66871872422197194573137055386260077430224454921744173428...  
1  73268068968476052541700461224016254385370856675658988668...  
2  96267458254090441798245236969531476202789337333051349091...  
3  38668657318576350037621534664851279396445838979397040424...  
4  75106699590274454429918698609373837985540110268405910990...  


In [58]:
#@title 3 - Fetch price histories

def fetch_price_history(token_id, resolution_time, hours_before=48):
    """Fetch price history - note: closed markets only support 12h+ granularity."""

    if isinstance(resolution_time, str):
        res_time = datetime.fromisoformat(resolution_time.replace("Z", "+00:00"))
    else:
        res_time = resolution_time

    start_time = res_time - timedelta(hours=hours_before)

    # correct host: clob not gamma-api
    url = "https://clob.polymarket.com/prices-history"
    params = {
        "market":    token_id,
        "interval":  "max",
        "fidelity":  720,   # 12 hours in minutes - minimum for closed markets
        "startTs":   int(start_time.timestamp()),
        "endTs":     int(res_time.timestamp()),
    }

    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()

        if not data or "history" not in data or not data["history"]:
            return pd.DataFrame()

        # columns are 't' and 'p' not 'timestamp' and 'price'
        df = pd.DataFrame(data["history"])
        df = df.rename(columns={"t": "timestamp", "p": "price"})
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
        df["price"] = df["price"].astype(float)
        return df

    except Exception as e:
        print(f"  Error: {e}")
        return pd.DataFrame()

# Test on first 5 markets
for _, row in df_markets.head(5).iterrows():
    history = fetch_price_history(row["token_id"], row["resolution_time"])
    print(f"Market: {row['question'][:55]}...")
    print(f"  Points: {len(history)}  |  Volume: ${row['volume']:,.0f}")
    if len(history) > 0:
        print(f"  Price: {history['price'].iloc[0]:.2f} → {history['price'].iloc[-1]:.2f}")
    print()

Market: idOS FDV above $100M one day after launch?...
  Points: 6  |  Volume: $9,606
  Price: 0.39 → 0.00

Market: idOS FDV above $50M one day after launch?...
  Points: 6  |  Volume: $21,938
  Price: 0.46 → 0.02

Market: Espresso FDV above $50M one day after launch?...
  Points: 8  |  Volume: $83,280
  Price: 0.99 → 1.00

Market: Espresso FDV above $1B one day after launch?...
  Points: 7  |  Volume: $79,468
  Price: 0.02 → 0.00

Market: idOS FDV above $80M one day after launch?...
  Points: 6  |  Volume: $12,134
  Price: 0.41 → 0.01



In [59]:
#@title 4 - Compute VPIN and price features

import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def compute_vpin_from_prices(history_df):
    if len(history_df) < 3:
        return None
    prices = history_df["price"].values
    changes = np.diff(prices)
    buy_pressure = np.where(changes > 0, changes, 0).sum()
    sell_pressure = np.where(changes < 0, -changes, 0).sum()
    total = buy_pressure + sell_pressure
    if total == 0:
        return 0.0
    return abs(buy_pressure - sell_pressure) / total

def compute_price_features(history_df):
    if len(history_df) < 2:
        return None
    prices = history_df["price"].values
    changes = np.abs(np.diff(prices))
    return {
        "price_volatility": changes.std() if len(changes) > 1 else 0,
        "max_single_move": changes.max() if len(changes) > 0 else 0,
        "final_price": prices[-1],
        "starting_price": prices[0],
        "total_price_move": abs(prices[-1] - prices[0]),
    }

histories = {}  # NEW — cache for Cell 4b

results = []
for _, row in df_markets.iterrows():
    history = fetch_price_history(row["token_id"], row["resolution_time"])
    if len(history) < 3:
        continue

    starting_price = history["price"].iloc[0]
    if not (0.15 <= starting_price <= 0.85):
        continue

    histories[row["token_id"]] = history  # NEW — store history for reuse

    vpin = compute_vpin_from_prices(history)
    features = compute_price_features(history)
    if vpin is None or features is None:
        continue

    results.append({
        "question": row["question"],
        "volume": row["volume"],
        "vpin_score": vpin,
        **features,
    })

df_scored = pd.DataFrame(results)
print(f"Markets with enough data: {len(df_scored)}")

feature_cols = ["vpin_score", "volume", "total_price_move", "price_volatility", "max_single_move"]
X = df_scored[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model = IsolationForest(contamination=0.1, random_state=42)
df_scored["anomaly_score"] = model.fit_predict(X_scaled)
df_scored["suspicion_score"] = -model.decision_function(X_scaled)

top = df_scored.nlargest(15, "suspicion_score")[
    ["question", "suspicion_score", "vpin_score", "volume", "starting_price", "final_price"]
].reset_index(drop=True)
print(top.to_string())

Markets with enough data: 132
                                               question  suspicion_score  vpin_score      volume  starting_price  final_price
0            Stable FDV above $2B one day after launch?            0.139       0.269 2108189.722           0.770        0.002
1   Over $500K committed to the Sol Raffle public sale?            0.110       0.988   45128.015           0.765        0.007
2           Owlto FDV above $200M one day after launch?            0.082       0.588   42670.776           0.205        0.992
3          Seeker FDV above $100M one day after launch?            0.081       0.013  314602.962           0.730        0.755
4             Trove FDV above $5M one day after launch?            0.073       1.000  234777.441           0.815        0.001
5           Lighter FDV above $3B one day after launch?            0.056       0.880 1203029.902           0.530        0.001
6         Kinetiq FDV above $250M one day after launch?            0.044       0.318  83

In [60]:
#@title 5 - Compute resolution surprise features

def compute_resolution_surprise(history_df):
    if len(history_df) < 2:
        return None, None
    prices = history_df["price"].values
    actual_resolution = 1.0 if prices[-1] > 0.5 else 0.0
    prior_price = prices[0]
    surprise_score = abs(actual_resolution - prior_price)
    total_move = abs(prices[-1] - prices[0])
    final_step_move = abs(prices[-1] - prices[-2])
    late_move_ratio = (final_step_move / total_move) if total_move > 0.01 else 0.0
    return surprise_score, late_move_ratio

surprise_scores = []
late_move_ratios = []

for _, row in df_scored.iterrows():
    token_id = df_markets.loc[
        df_markets["question"] == row["question"], "token_id"
    ].values[0]
    history = histories.get(token_id)
    surprise, late_move = compute_resolution_surprise(history) if history is not None else (0.0, 0.0)
    surprise_scores.append(surprise)
    late_move_ratios.append(late_move)

df_scored["surprise_score"] = surprise_scores
df_scored["late_move_ratio"] = late_move_ratios

print(df_scored[["question", "surprise_score", "late_move_ratio"]].head(10).to_string())

                                            question  surprise_score  late_move_ratio
0         idOS FDV above $100M one day after launch?           0.385            0.038
1          idOS FDV above $50M one day after launch?           0.455            0.160
2          idOS FDV above $80M one day after launch?           0.405            0.027
3     Espresso FDV above $200M one day after launch?           0.550            0.845
4     Espresso FDV above $400M one day after launch?           0.265            0.011
5         idOS FDV above $200M one day after launch?           0.230            0.105
6     Espresso FDV above $300M one day after launch?           0.480            0.014
7          idOS FDV above $20M one day after launch?           0.410            0.425
8       ETHGAS FDV above $500M one day after launch?           0.305            0.195
9  Will Hyperliquid dip to $28 by December 31, 2026?           0.365            0.104


In [61]:
#@title  6 - Compute time-weighted VPIN

def compute_time_weighted_vpin(history_df):
    """
    Like VPIN, but movements closer to resolution are weighted more heavily.
    Weight increases linearly: first move = 1, last move = N (number of steps).
    """
    if len(history_df) < 3:
        return None

    prices = history_df["price"].values
    changes = np.diff(prices)
    n = len(changes)

    # Linear weights: 1, 2, 3 ... N (last move gets highest weight)
    weights = np.arange(1, n + 1, dtype=float)

    weighted_buy = np.where(changes > 0, changes * weights, 0).sum()
    weighted_sell = np.where(changes < 0, -changes * weights, 0).sum()
    total = weighted_buy + weighted_sell

    if total == 0:
        return 0.0
    return abs(weighted_buy - weighted_sell) / total

# Add to df_scored
tw_vpins = []
for _, row in df_scored.iterrows():
    token_id = df_markets.loc[
        df_markets["question"] == row["question"], "token_id"
    ].values[0]
    history = histories.get(token_id)
    tw_vpin = compute_time_weighted_vpin(history) if history is not None else 0.0
    tw_vpins.append(tw_vpin)

df_scored["time_weighted_vpin"] = tw_vpins
print(df_scored[["question", "vpin_score", "time_weighted_vpin"]].head(10).to_string())

                                            question  vpin_score  time_weighted_vpin
0         idOS FDV above $100M one day after launch?       1.000               1.000
1          idOS FDV above $50M one day after launch?       1.000               1.000
2          idOS FDV above $80M one day after launch?       1.000               1.000
3     Espresso FDV above $200M one day after launch?       0.466               0.435
4     Espresso FDV above $400M one day after launch?       0.726               0.690
5         idOS FDV above $200M one day after launch?       0.746               0.702
6     Espresso FDV above $300M one day after launch?       0.906               0.919
7          idOS FDV above $20M one day after launch?       1.000               1.000
8       ETHGAS FDV above $500M one day after launch?       0.442               0.348
9  Will Hyperliquid dip to $28 by December 31, 2026?       0.669               0.686


In [62]:
#@title 7 - Score markets with Isolation Forest

feature_cols = [
    "vpin_score", "volume", "total_price_move",
    "price_volatility", "max_single_move",
    "surprise_score", "late_move_ratio"
]
X = df_scored[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model = IsolationForest(contamination=0.1, random_state=42)
df_scored["anomaly_score"] = model.fit_predict(X_scaled)
df_scored["suspicion_score"] = -model.decision_function(X_scaled)

top = df_scored.nlargest(15, "suspicion_score")[
    ["question", "suspicion_score", "vpin_score", "surprise_score",
     "late_move_ratio", "volume", "starting_price", "final_price"]
].reset_index(drop=True)
print(top.to_string())


                                               question  suspicion_score  vpin_score  surprise_score  late_move_ratio      volume  starting_price  final_price
0          Seeker FDV above $100M one day after launch?            0.206       0.013           0.270           18.400  314602.962           0.730        0.755
1            Stable FDV above $2B one day after launch?            0.141       0.269           0.770            0.226 2108189.722           0.770        0.002
2   Over $500K committed to the Sol Raffle public sale?            0.088       0.988           0.765            0.006   45128.015           0.765        0.007
3             Trove FDV above $5M one day after launch?            0.085       1.000           0.815            0.079  234777.441           0.815        0.001
4           Owlto FDV above $200M one day after launch?            0.074       0.588           0.795            0.716   42670.776           0.205        0.992
5         Over $20M committed to the Space pub

In [64]:
#@title 8 - Export and download results

df_scored.nlargest(30, "suspicion_score").to_csv("polymarket_suspicious.csv", index=False)
from google.colab import files
#skip for now
# files.download("polymarket_suspicious.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [65]:
#@title 9 - Preview results

print(df_scored.nlargest(30, "suspicion_score").to_csv(index=False))

question,volume,vpin_score,price_volatility,max_single_move,final_price,starting_price,total_price_move,anomaly_score,suspicion_score,surprise_score,late_move_ratio,time_weighted_vpin
Seeker FDV above $100M one day after launch?,314602.962484,0.013123359580052504,0.1206119058547291,0.46,0.755,0.73,0.025000000000000022,-1,0.2059357356351944,0.27,18.399999999999984,0.006289308176100654
Stable FDV above $2B one day after launch?,2108189.721855,0.26884729753367154,0.06717622875598588,0.56,0.0015,0.77,0.7685000000000001,-1,0.1414396162946141,0.77,0.22576447625243978,0.45790710972742876
Over $500K committed to the Sol Raffle public sale?,45128.015107,0.9882583170254404,0.19638844638878325,0.497,0.0075,0.765,0.7575000000000001,-1,0.08799145519780294,0.765,0.00594059405940594,0.9800995024875622
Trove FDV above $5M one day after launch?,234777.440695,1.0,0.14214654410150107,0.3949999999999999,0.0005,0.815,0.8145,-1,0.08503069560099341,0.815,0.07918968692449356,1.0
Owlto FDV above $200M one day 